# 📷 OCR Interactivo — Versión Pedagógica para Diplomado en Visión Artificial

---

## 🎯 ¿Qué aprenderás en este notebook?

Este cuaderno está diseñado para que **aprendas leyendo el código mismo**.  
Cada línea tiene una explicación clara de:

- **¿Qué hace?** → la acción concreta
- **¿Por qué se hace?** → la razón técnica
- **¿Qué pasaría si se omite?** → el impacto de cada decisión

---

## 📚 Conceptos clave que cubre este notebook

| Concepto | Descripción |
|---|---|
| **OCR** | Reconocimiento Óptico de Caracteres — convierte imágenes en texto |
| **Tesseract** | Motor de OCR de código abierto mantenido por Google |
| **OpenCV** | Biblioteca de visión artificial para procesar imágenes con Python |
| **Preprocesamiento** | Mejorar la imagen ANTES de aplicar OCR para obtener mejores resultados |
| **Umbral (Threshold)** | Convertir la imagen a blanco/negro para que Tesseract lea mejor el texto |
| **PSM** | Page Segmentation Mode — cómo Tesseract interpreta la estructura del texto |
| **ipywidgets** | Controles interactivos (botones, sliders) dentro de Jupyter Notebook |

---

## 🔄 Flujo general del programa

```
IMAGEN DE ENTRADA
       ↓
PREPROCESAMIENTO (3 métodos en paralelo)
   ├── Sin cambios (solo grises)
   ├── Umbral Otsu (global)
   └── Umbral Adaptativo (local)
       ↓
OCR CON TESSERACT (para cada versión)
       ↓
SELECCIÓN AUTOMÁTICA DEL MEJOR MÉTODO
       ↓
VISUALIZACIÓN + TEXTO EXTRAÍDO + EXPORTACIÓN
```

---

> ✅ **Instrucción**: Ejecuta las celdas **en orden** con `Shift + Enter`.  
> 🔁 Si reinicias el kernel, empieza desde la celda 1.

---
## 📦 CELDA 1 — Instalación de librerías

### ¿Qué son las librerías?
Una **librería** (o biblioteca) es un conjunto de código ya escrito por otras personas  
que puedes reutilizar sin tener que programarlo desde cero.

| Librería | ¿Para qué sirve en este proyecto? |
|---|---|
| `pytesseract` | Interfaz Python para hablar con el motor Tesseract-OCR |
| `pillow` | Leer, guardar y convertir imágenes (PNG, JPG, BMP...) |
| `opencv-python` | Procesamiento avanzado de imágenes: filtros, umbrales, detección |
| `matplotlib` | Mostrar imágenes y gráficas dentro del notebook |
| `pandas` | Organizar los datos del OCR en tablas (filas y columnas) |
| `ipywidgets` | Crear controles interactivos: botones, cuadros de texto, etc. |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — INSTALACIÓN DE LIBRERÍAS                         ║
# ╚══════════════════════════════════════════════════════════════╝
#
# El símbolo % al inicio indica que es un 'comando mágico' de Jupyter.
# %pip es equivalente a escribir 'pip install ...' en la terminal,
# pero funciona directamente desde el notebook.
#
# La opción -q (quiet) suprime los mensajes de instalación para que
# la salida sea más limpia. Si quieres ver el detalle, quita el -q.
#
# ⚠️ IMPORTANTE: Solo necesitas ejecutar esta celda UNA VEZ por sesión.
# Si ya tienes las librerías instaladas, Python simplemente confirmará
# que ya están disponibles y no hará nada más.

%pip install -q pytesseract pillow opencv-python matplotlib pandas ipywidgets

# Esta línea solo sirve para darte retroalimentación visual:
# Si ves este mensaje, la instalación fue exitosa.
print("✅ Librerías instaladas correctamente.")

---
## 📥 CELDA 2 — Importaciones y configuración de Tesseract

### ¿Instalar vs Importar?
- **Instalar** (celda anterior): descarga la librería a tu computadora. Se hace una vez.
- **Importar** (esta celda): le dice a Python *"usa esta librería ahora mismo"*. Se hace cada vez que ejecutas el programa.

### ¿Qué es Tesseract?
Tesseract es un **programa separado** que Python no puede usar directamente.  
El puente entre Python y Tesseract es la librería `pytesseract`.  
Para que ese puente funcione, Python necesita saber **dónde está instalado Tesseract**.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — IMPORTACIONES Y CONFIGURACIÓN                    ║
# ╚══════════════════════════════════════════════════════════════╝

# --- IMPORTACIONES ESTÁNDAR DE PYTHON ---
# Estas librerías vienen incluidas con Python (no requieren instalación):

import os     # Permite interactuar con el sistema operativo:
               # verificar si un archivo existe, listar carpetas, etc.

import io     # Permite trabajar con flujos de datos en memoria
               # (como si fueran archivos, pero sin guardarlos en disco).
               # Lo usamos para convertir bytes de imagen a objetos PIL.

# --- IMPORTACIONES DE VISIÓN ARTIFICIAL ---

import cv2    # OpenCV: la biblioteca más usada en visión artificial.
               # Permite leer, transformar, filtrar y analizar imágenes.
               # 'cv2' es el nombre con el que se importa (viene de OpenCV 2).

import numpy as np
               # NumPy: manejo de arrays numéricos (matrices).
               # Una imagen digital ES una matriz: filas x columnas x canales de color.
               # 'as np' crea un alias corto para no escribir 'numpy' cada vez.

import pytesseract
               # Puente entre Python y el motor OCR Tesseract.
               # Convierte imágenes en texto y datos estructurados.

from PIL import Image
               # PIL (Python Imaging Library), también conocida como Pillow.
               # 'from ... import ...' trae solo la clase Image (no toda la librería).
               # Image permite abrir archivos JPG, PNG, BMP y convertir formatos.

# --- IMPORTACIONES PARA VISUALIZACIÓN ---

import matplotlib.pyplot as plt
               # matplotlib es la biblioteca estándar para graficar en Python.
               # 'pyplot' es el submódulo que da funciones estilo MATLAB.
               # plt.imshow() muestra imágenes; plt.show() las renderiza.

import matplotlib.patches as patches
               # patches permite dibujar formas geométricas en gráficas:
               # rectángulos, círculos, flechas, etc.
               # Lo usamos para la leyenda de colores de confianza.

# --- IMPORTACIONES PARA DATOS Y UI ---

import pandas as pd
               # pandas: manejo de datos tabulares (como Excel en Python).
               # Los resultados de OCR de Tesseract vienen en formato DataFrame (tabla).

import ipywidgets as widgets
               # ipywidgets: crea elementos interactivos en Jupyter:
               # botones, barras de progreso, subidores de archivos, etc.

from IPython.display import display, clear_output
               # display()       → muestra cualquier objeto (widget, imagen, tabla)
               # clear_output()  → borra el contenido anterior de una celda de salida
               #                   (para actualizar resultados sin duplicarlos)


# ════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE LA RUTA DE TESSERACT
# ════════════════════════════════════════════════════════════════
#
# Tesseract es un programa externo (no es Python).
# pytesseract necesita saber dónde está su ejecutable (.exe en Windows).
#
# La 'r' antes de la cadena indica 'raw string':
# evita que las barras invertidas \ se interpreten como escapes (\n, \t, etc.).
# Sin la 'r', tendrías que escribir: "C:\\Program Files\\Tesseract-OCR\\tesseract.exe"

RUTA_TESSERACT = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# os.path.exists() devuelve True si el archivo existe, False si no.
# Así verificamos antes de configurar, para dar un mensaje útil al usuario.

if os.path.exists(RUTA_TESSERACT):
    # Si el archivo existe, lo asignamos como ruta del ejecutable.
    # pytesseract.pytesseract es el módulo interno; tesseract_cmd es la variable
    # que guarda la ruta al programa.
    pytesseract.pytesseract.tesseract_cmd = RUTA_TESSERACT

    print("✅ Tesseract encontrado correctamente.")

    # get_tesseract_version() llama al programa Tesseract y pide su versión.
    # Útil para confirmar que la comunicación Python ↔ Tesseract funciona.
    print(f"Versión detectada: {pytesseract.get_tesseract_version()}")

else:
    # Si no se encontró, informamos al estudiante qué ajustar.
    print("⚠️  No se encontró Tesseract en la ruta configurada.")
    print(f"Ruta actual: {RUTA_TESSERACT}")
    print("Modifica la variable RUTA_TESSERACT y vuelve a ejecutar esta celda.")
    print()
    print("📌 Dónde instalar Tesseract:")
    print("   Windows → https://github.com/UB-Mannheim/tesseract/wiki")
    print("   macOS   → brew install tesseract")
    print("   Linux   → sudo apt install tesseract-ocr")


# ════════════════════════════════════════════════════════════════
# VARIABLES GLOBALES
# ════════════════════════════════════════════════════════════════
#
# Las variables 'globales' son accesibles desde CUALQUIER función
# del programa. Aquí las inicializamos para que siempre existan,
# aunque aún no tengan datos reales.
#
# Python NO tiene variables globales 'automáticas': si no se declaran
# antes de usarlas en una función, el programa lanza un NameError.

ultima_imagen_rgb  = None          # Guardará la última imagen procesada como array RGB.
                                   # None = "vacío / sin valor aún".

ultimo_texto       = ""            # Guardará el texto extraído por OCR.
                                   # Se inicializa como cadena vacía.

ultimo_metodo      = ""            # Guardará el nombre del mejor método de preprocesamiento
                                   # ('ninguno', 'otsu' o 'adaptativo').

ultimos_datos      = pd.DataFrame()  # DataFrame vacío para los datos de palabras detectadas.
                                     # pd.DataFrame() crea una tabla sin filas ni columnas.

ultimo_nombre_archivo = ""         # Guardará el nombre del archivo de imagen subido.

print("✅ Variables globales inicializadas.")

---
## 🔧 CELDA 3 — Funciones de preprocesamiento y OCR

### ¿Por qué preprocesar la imagen antes del OCR?

Tesseract fue diseñado para trabajar con texto **claro sobre fondo blanco**.  
En el mundo real, las imágenes tienen:
- 📷 Variaciones de iluminación
- 🌑 Sombras y brillos
- 🎨 Fondos de colores
- 🔇 Ruido (píxeles aleatorios)

**El preprocesamiento** elimina esas imperfecciones para que Tesseract "vea mejor".

### Los 3 métodos que compararemos:

```
Imagen a color
      ↓
  Escala de grises (elimina color, mantiene intensidad)
      ↓
  ┌──────────────────────────────────────────┐
  │ Método 1: NINGUNO   → Solo grises        │
  │ Método 2: OTSU      → Umbral automático  │  
  │ Método 3: ADAPTATIVO→ Umbral local       │
  └──────────────────────────────────────────┘
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — FUNCIONES DE PREPROCESAMIENTO Y OCR              ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Una FUNCIÓN es un bloque de código reutilizable.
# Se define con 'def nombre(parámetros):' y se ejecuta llamándola: nombre(valor)
#
# Ventajas de usar funciones:
#   1. Escribes el código una vez y lo reutilizas muchas veces
#   2. Si hay un error, solo corriges en un lugar
#   3. El código queda organizado y legible


# ──────────────────────────────────────────────────────────────
# FUNCIÓN 1: preprocesar_imagen
# ──────────────────────────────────────────────────────────────
# PROPÓSITO: Tomar una imagen en color y convertirla a blanco/negro
#            para mejorar la precisión del OCR.
#
# PARÁMETROS:
#   img_np  → la imagen como array NumPy (matriz de números)
#   metodo  → qué técnica usar: 'ninguno', 'otsu' o 'adaptativo'
#              El valor por defecto es 'adaptativo' (mejor para imágenes reales)
#
# RETORNA: imagen preprocesada como array NumPy en escala de grises

def preprocesar_imagen(img_np, metodo='adaptativo'):

    # ── PASO 1: Convertir a escala de grises ──────────────────
    #
    # Una imagen color tiene 3 capas: Rojo, Verde, Azul (RGB)
    # → forma del array: (alto, ancho, 3)
    #
    # Una imagen en grises tiene 1 capa con valores de 0 (negro) a 255 (blanco)
    # → forma del array: (alto, ancho)
    #
    # len(img_np.shape) == 3 comprueba si la imagen tiene 3 dimensiones (es color).
    # Si es 2 (ya en grises), la usamos directamente.

    if len(img_np.shape) == 3:      # Si la imagen es de color (3 canales RGB)
        gris = cv2.cvtColor(         # cv2.cvtColor() convierte entre espacios de color
            img_np,                  #   → imagen de entrada
            cv2.COLOR_RGB2GRAY       #   → código de conversión: de RGB a Gris
        )                            # Resultado: array 2D con intensidades 0-255
    else:
        gris = img_np.copy()         # Ya es gris: hacemos una copia para no modificar
                                     # el original (buena práctica en visión artificial)

    # ── CASO ESPECIAL: método 'ninguno' ────────────────────────
    # Si el usuario eligió 'ninguno', devolvemos la imagen en grises
    # sin ningún procesamiento adicional. Sirve como línea base de comparación.

    if metodo == 'ninguno':
        return gris                  # 'return' termina la función y devuelve el valor

    # ── PASO 2: Escalar imagen pequeña ────────────────────────
    #
    # Tesseract funciona MUCHO mejor con imágenes de al menos 300 DPI.
    # Si la imagen es muy pequeña (ancho < 600 px), la ampliamos x2.
    #
    # img.shape devuelve (alto, ancho) para imágenes en grises.
    # Usamos 'h, w' (height, width) como nombres de variables.

    h, w = gris.shape                # Desempaquetamos las dimensiones

    if w < 600:                      # Si el ancho es menor a 600 píxeles
        gris = cv2.resize(
            gris,
            (w * 2, h * 2),          # Nuevo tamaño: el doble en ambas dimensiones
            interpolation=cv2.INTER_CUBIC
                                     # INTER_CUBIC: interpolación bicúbica
                                     # Mejor calidad que INTER_LINEAR al ampliar
                                     # (calcula píxeles nuevos de forma más suave)
        )

    # ── PASO 3: Suavizado Gaussiano ────────────────────────────
    #
    # El ruido (píxeles aleatorios) confunde al OCR.
    # GaussianBlur aplica un desenfoque ligero para eliminar ruido de alta frecuencia.
    #
    # Parámetros:
    #   gris     → imagen de entrada
    #   (3, 3)   → tamaño del kernel (ventana de 3x3 píxeles)
    #              Kernels más grandes = más desenfoque. Para OCR, 3x3 es suficiente.
    #   0        → sigma (desviación estándar): 0 = calcular automáticamente según kernel

    suavizado = cv2.GaussianBlur(gris, (3, 3), 0)

    # ── PASO 4: Umbralización (Thresholding) ──────────────────
    #
    # Umbralizar = convertir a imagen BINARIA (solo 0=negro o 255=blanco)
    # El objetivo: texto negro sobre fondo blanco (ideal para Tesseract)

    if metodo == 'otsu':
        # ─── MÉTODO OTSU ──────────────────────────────────────
        # Otsu calcula AUTOMÁTICAMENTE el mejor umbral global.
        # Analiza el histograma de la imagen y encuentra el valor
        # que mejor separa píxeles de texto (oscuro) del fondo (claro).
        #
        # cv2.threshold() devuelve TUPLA: (valor_umbral, imagen_binaria)
        # El '_' descarta el primer valor (valor numérico del umbral)
        # ya que solo nos interesa la imagen resultante.
        #
        # Flags:
        #   cv2.THRESH_BINARY → píxeles > umbral = 255, demás = 0
        #   cv2.THRESH_OTSU   → activa el cálculo automático de Otsu
        #   El '+'  combina ambos flags con OR binario

        _, procesada = cv2.threshold(
            suavizado,
            0,                                          # umbral inicial (Otsu lo reemplaza)
            255,                                        # valor máximo para píxeles que superan el umbral
            cv2.THRESH_BINARY + cv2.THRESH_OTSU         # combinación de flags
        )

    elif metodo == 'adaptativo':
        # ─── MÉTODO ADAPTATIVO ────────────────────────────────
        # A diferencia de Otsu (umbral único para toda la imagen),
        # el umbral adaptativo calcula un umbral DIFERENTE para cada
        # pequeña región de la imagen.
        #
        # ¿Por qué es mejor en muchos casos?
        # Si la imagen tiene sombras o iluminación desigual,
        # un umbral global puede perder texto en zonas oscuras o claras.
        # El adaptativo se ajusta localmente a cada zona.

        procesada = cv2.adaptiveThreshold(
            suavizado,                              # imagen de entrada (suavizada)
            255,                                    # valor máximo para píxeles de texto
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,         # el umbral local se calcula como
                                                    # promedio ponderado (Gaussiano) de la zona
            cv2.THRESH_BINARY,                      # resultado binario: 0 o 255
            11,                                     # blockSize: tamaño de la región local (11x11 px)
                                                    # debe ser impar y > 1
            2                                       # C: constante que se resta al promedio
                                                    # Ajusta la sensibilidad del umbral local
        )

    else:
        # Si el método no es ninguno de los anteriores,
        # devolvemos la imagen suavizada sin umbralizar
        procesada = gris

    return procesada   # Devolvemos la imagen preprocesada lista para el OCR


# ──────────────────────────────────────────────────────────────
# FUNCIÓN 2: ejecutar_ocr
# ──────────────────────────────────────────────────────────────
# PROPÓSITO: Enviar la imagen preprocesada a Tesseract y obtener:
#   1. El texto completo como cadena de texto
#   2. Una tabla con cada palabra detectada + su posición y confianza
#
# PARÁMETROS:
#   img_np  → imagen como array NumPy (puede ser grises o color)
#   idioma  → idiomas para reconocer: 'spa' = español, 'eng' = inglés
#              'spa+eng' = ambos (útil para textos mixtos)
#   psm     → Page Segmentation Mode (ver tabla en la celda 6)
#              3 = modo automático, el más común para páginas completas

def ejecutar_ocr(img_np, idioma='spa+eng', psm=3):

    # Tesseract trabaja con objetos PIL Image, no con arrays NumPy.
    # Image.fromarray() convierte el array NumPy a objeto PIL.
    # Es un paso obligatorio para usar pytesseract.
    img_pil = Image.fromarray(img_np)

    # Construimos la cadena de configuración para Tesseract.
    # Formato: '--psm NUMERO -l IDIOMA'
    # El f-string (f'...') permite insertar variables dentro de cadenas con {}
    config = f'--psm {psm} -l {idioma}'

    # image_to_string() envía la imagen a Tesseract y devuelve el texto
    # como una sola cadena con saltos de línea (\n) donde hay párrafos.
    texto = pytesseract.image_to_string(img_pil, config=config)

    # image_to_data() devuelve NO SOLO el texto, sino datos detallados:
    # para cada palabra: posición (x, y), tamaño (ancho, alto), nivel de confianza.
    # output_type=DATAFRAME hace que la respuesta sea directamente un DataFrame de pandas.
    datos = pytesseract.image_to_data(
        img_pil,
        config=config,
        output_type=pytesseract.Output.DATAFRAME  # retorna tabla pandas en lugar de string
    )

    # ── Limpieza del DataFrame ─────────────────────────────────
    #
    # Tesseract incluye filas vacías (cuando no detectó texto en una región).
    # Las limpiamos para quedarnos solo con palabras reales.

    # fillna('') reemplaza valores NaN (vacíos) por cadena vacía ''
    # .astype(str) asegura que todos los valores sean strings (evita errores de tipo)
    datos['text'] = datos['text'].fillna('').astype(str)

    # Filtramos filas que:
    #   - conf > 0   → tienen confianza positiva (−1 significa "no es texto")
    #   - text != '' → no están vacías (strip() elimina espacios en blanco)
    # .copy() crea una copia independiente (evita el SettingWithCopyWarning de pandas)
    datos_df = datos[
        (datos['conf'].fillna(-1) > 0) &          # solo filas con confianza válida
        (datos['text'].str.strip() != '')          # solo filas con texto no vacío
    ][['text', 'left', 'top', 'width', 'height', 'conf']].copy()
    #  ↑ Seleccionamos solo las columnas que nos interesan:
    #    text = la palabra detectada
    #    left, top = coordenadas del borde superior izquierdo del rectángulo
    #    width, height = dimensiones del rectángulo alrededor de la palabra
    #    conf = confianza del OCR (0-100): qué tan seguro está Tesseract

    return texto, datos_df   # Devolvemos tanto el texto completo como la tabla detallada


# ──────────────────────────────────────────────────────────────
# FUNCIÓN 3: dibujar_resultados
# ──────────────────────────────────────────────────────────────
# PROPÓSITO: Visualizar la imagen original junto a otra imagen
#            donde se dibujan rectángulos alrededor de cada palabra detectada.
#            Los colores indican el nivel de confianza del OCR:
#              🟢 Verde  → confianza alta  (> 80%)
#              🟠 Naranja → confianza media (50–80%)
#              🔴 Rojo   → confianza baja  (< 50%)

def dibujar_resultados(img_original, datos_df, titulo='Texto detectado'):

    # Si la imagen es en grises (2D), la convertimos a RGB (3D) para poder
    # dibujar rectángulos de colores encima (en grises solo hay gris).
    if len(img_original.shape) == 2:
        img_base = cv2.cvtColor(img_original, cv2.COLOR_GRAY2RGB)
    else:
        img_base = img_original.copy()   # copia para no modificar el original

    img_anotada = img_base.copy()        # segunda copia donde dibujaremos

    # ── Dibujar rectángulo por cada palabra ────────────────────
    #
    # iterrows() recorre el DataFrame fila por fila.
    # Cada fila es una palabra detectada con su posición y confianza.
    # El '_' descarta el índice de fila (no lo necesitamos).

    for _, fila in datos_df.iterrows():

        # Extraemos las coordenadas del rectángulo y las convertimos a enteros
        # (int) porque cv2 no acepta flotantes en coordenadas de píxeles.
        x, y = int(fila['left']), int(fila['top'])    # esquina superior izquierda
        w, h = int(fila['width']), int(fila['height'])  # ancho y alto del rectángulo
        conf = float(fila['conf'])                     # confianza como número decimal

        # Asignamos el color según el nivel de confianza.
        # Los colores son tuplas (R, G, B) con valores 0-255.
        # Este es un ejemplo de expresión condicional anidada (ternario en Python):
        # valor_si_verdad if condición else valor_si_falso
        color = (0, 200, 0) if conf > 80 else (255, 165, 0) if conf > 50 else (220, 0, 0)
        #         Verde            alta            Naranja       media          Rojo    baja

        # cv2.rectangle() dibuja un rectángulo (no relleno) sobre la imagen.
        # Parámetros: imagen, esquina_sup_izq, esquina_inf_der, color, grosor_linea
        cv2.rectangle(img_anotada, (x, y), (x + w, y + h), color, 2)

        # cv2.putText() escribe el texto de la palabra sobre la imagen.
        # max(15, y - 5) asegura que el texto no quede fuera de la imagen
        # si la palabra está en el borde superior (y < 5).
        cv2.putText(
            img_anotada,
            str(fila['text']),              # el texto de la palabra
            (x, max(15, y - 5)),            # posición: justo arriba del rectángulo
            cv2.FONT_HERSHEY_SIMPLEX,       # fuente estándar de OpenCV
            0.5,                            # tamaño de fuente (escala)
            color,                          # mismo color que el rectángulo
            1,                              # grosor del texto
            cv2.LINE_AA                     # antialiasing: bordes más suaves
        )

    # ── Mostrar las dos imágenes una al lado de la otra ────────
    #
    # plt.subplots(1, 2) crea una figura con 1 fila y 2 columnas de subgráficas.
    # figsize=(16, 7) define el tamaño en pulgadas: ancho=16, alto=7.
    # axes es una lista con los 2 ejes (subgráficas).

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    axes[0].imshow(img_base)                    # imagen sin anotaciones
    axes[0].set_title('📷 Imagen original')     # título del panel izquierdo
    axes[0].axis('off')                          # oculta los ejes X e Y

    axes[1].imshow(img_anotada)                  # imagen con rectángulos dibujados
    axes[1].set_title(f'🔍 {titulo}')            # título con f-string
    axes[1].axis('off')

    # ── Leyenda de colores ─────────────────────────────────────
    # patches.Patch crea un rectángulo de color para la leyenda.
    # Pasamos una lista de parches a legend() para que los muestre.

    leyenda = [
        patches.Patch(color='#00c800', label='Confianza alta  (> 80%)'),
        patches.Patch(color='#ffa500', label='Confianza media (50–80%)'),
        patches.Patch(color='#dc0000', label='Confianza baja  (< 50%)'),
    ]
    axes[1].legend(handles=leyenda, loc='lower left', fontsize=9)

    plt.tight_layout()   # ajusta automáticamente el espaciado entre subgráficas
    plt.show()           # renderiza y muestra la figura en el notebook


# ──────────────────────────────────────────────────────────────
# FUNCIÓN 4: mostrar_resultados_ocr
# ──────────────────────────────────────────────────────────────
# PROPÓSITO: Imprimir en consola el texto extraído y la tabla de
#            palabras con sus niveles de confianza, de mayor a menor.

def mostrar_resultados_ocr(texto, datos_df):

    # Imprimimos una línea de separación visual
    print("\n" + "=" * 55)
    print("  📝 TEXTO EXTRAÍDO")
    print("=" * 55)

    # .strip() elimina espacios y saltos de línea al inicio y final del texto.
    # Si después de strip() el texto no está vacío → lo imprimimos.
    # Si está vacío → informamos que no se detectó nada.
    print(texto.strip() if texto.strip() else "⚠️ No se detectó texto legible.")

    # Solo mostramos la tabla si hay datos (el DataFrame no está vacío)
    if not datos_df.empty:

        print("\n" + "=" * 55)
        print("  📊 PALABRAS DETECTADAS")
        print("=" * 55)

        # sort_values('conf', ascending=False) ordena de mayor a menor confianza
        # reset_index(drop=True) reinicia la numeración de filas desde 0
        tabla = datos_df.sort_values('conf', ascending=False).reset_index(drop=True)

        # Renombramos las columnas para que sean más descriptivas al imprimir
        tabla.columns = ['Palabra', 'X', 'Y', 'Ancho', 'Alto', 'Confianza (%)']

        # round(1) redondea a 1 decimal: 87.3456 → 87.3
        tabla['Confianza (%)'] = tabla['Confianza (%)'].round(1)

        # to_string(index=False) convierte el DataFrame a texto sin mostrar el índice
        print(tabla[['Palabra', 'Confianza (%)']].to_string(index=False))

        # Estadísticas resumidas
        print(f"\n📈 Confianza promedio : {datos_df['conf'].mean():.1f}%")
        #                                                        ↑ :.1f formatea el
        #                                                          float con 1 decimal
        print(f"🔤 Total palabras     : {len(datos_df)}")

    print("=" * 55)


# ──────────────────────────────────────────────────────────────
# FUNCIÓN 5: ocr_mejor_metodo
# ──────────────────────────────────────────────────────────────
# PROPÓSITO: Probar los 3 métodos de preprocesamiento y determinar
#            automáticamente cuál produce mejor resultado.
#
# CRITERIO DE SELECCIÓN: el método que detecta más palabras
#                        con confianza superior al 70%.

def ocr_mejor_metodo(img_np, idioma='spa+eng', psm=3):

    resultados = {}   # Diccionario para guardar los resultados de cada método.
                      # Un diccionario almacena pares clave:valor → {'metodo': datos}

    # Iteramos sobre los 3 métodos disponibles.
    # En cada iteración, 'metodo' toma uno de los valores de la lista.
    for metodo in ['ninguno', 'otsu', 'adaptativo']:

        img_proc = preprocesar_imagen(img_np, metodo=metodo)   # preprocesar
        texto, datos = ejecutar_ocr(img_proc, idioma=idioma, psm=psm)   # ejecutar OCR

        # Calculamos el 'score': cantidad de palabras con confianza > 70%.
        # Si el DataFrame está vacío, el score es 0.
        # datos[datos['conf'] > 70] filtra solo las filas donde la confianza supera 70.
        # len() cuenta cuántas filas quedan después del filtro.
        score = len(datos[datos['conf'] > 70]) if not datos.empty else 0

        # Guardamos todos los resultados en el diccionario.
        # La clave es el nombre del método; el valor es otro diccionario con los datos.
        resultados[metodo] = {
            'img_proc': img_proc,   # imagen preprocesada (para mostrar comparación)
            'texto': texto,          # texto extraído
            'datos': datos,          # tabla de palabras
            'score': score           # puntuación de calidad
        }

    # max() con key=lambda encuentra el método con el mayor score.
    # lambda m: resultados[m]['score'] es una función anónima:
    # recibe un método 'm' y devuelve su score para comparar.
    metodo_ganador = max(resultados, key=lambda m: resultados[m]['score'])

    mejor = resultados[metodo_ganador]   # accedemos al resultado del método ganador

    # Devolvemos 4 valores a la vez (Python permite retornar tuplas)
    return mejor['texto'], metodo_ganador, resultados, mejor['datos']


# ──────────────────────────────────────────────────────────────
# FUNCIÓN 6: pipeline_ocr_completo
# ──────────────────────────────────────────────────────────────
# PROPÓSITO: Orquesta todo el proceso completo de principio a fin:
#   1. Prueba los 3 métodos
#   2. Selecciona el ganador
#   3. Muestra comparación visual de preprocesamiento
#   4. Dibuja los resultados
#   5. Imprime estadísticas
#   6. Guarda los resultados en variables globales
#
# Esta función es el 'director de orquesta' del programa.

def pipeline_ocr_completo(img_np, fuente='imagen', idioma='spa+eng', psm=3, mostrar_comparacion=True):

    # 'global' le dice a Python que estas variables viven FUERA de la función.
    # Sin 'global', Python crearía variables LOCALES con el mismo nombre,
    # que desaparecerían al terminar la función.
    global ultima_imagen_rgb, ultimo_texto, ultimo_metodo, ultimos_datos

    print(f"\n🔄 Ejecutando OCR ({fuente}) con 3 métodos de preprocesamiento...")

    # Llamamos a la función que prueba todos los métodos
    texto_mejor, metodo_ganador, resultados, datos_mejores = ocr_mejor_metodo(
        img_np, idioma=idioma, psm=psm
    )

    # Mostramos el score de cada método como tabla en consola.
    # f"{metodo:<12}" → alineación izquierda con 12 caracteres de ancho
    # f"{score:>3}"   → alineación derecha con 3 caracteres de ancho
    for metodo, info in resultados.items():   # .items() devuelve pares (clave, valor)
        print(f"   {metodo:<12} → {info['score']:>3} palabras con confianza > 70%")

    print(f"\n🏆 Mejor método automático: '{metodo_ganador}'")

    # ── Comparación visual de los 3 métodos ────────────────────
    if mostrar_comparacion:

        # Creamos una figura con 4 subgráficas en una fila:
        # Original | Solo grises | Otsu | Adaptativo
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        # Lista de tuplas: (imagen, título, mapa_de_color)
        # cmap=None → imagen a color; cmap='gray' → escala de grises
        imgs_vis = [
            (img_np,                               'Original (RGB)',      None),
            (resultados['ninguno']['img_proc'],    'Solo grises',         'gray'),
            (resultados['otsu']['img_proc'],       'Umbral Otsu',         'gray'),
            (resultados['adaptativo']['img_proc'], 'Umbral Adaptativo',   'gray'),
        ]

        # zip() combina dos iterables elemento por elemento.
        # Así axes[0] se empareja con imgs_vis[0], axes[1] con imgs_vis[1], etc.
        for ax, (img, titulo, cmap) in zip(axes, imgs_vis):
            ax.imshow(img, cmap=cmap)    # cmap=None para color, 'gray' para grises
            ax.set_title(titulo)
            ax.axis('off')

        plt.tight_layout()
        plt.show()

    # Dibujamos los resultados del mejor método
    dibujar_resultados(img_np, datos_mejores, titulo=f'OCR — método: {metodo_ganador}')

    # Imprimimos el resumen en texto
    mostrar_resultados_ocr(texto_mejor, datos_mejores)

    # ── Actualizar variables globales ──────────────────────────
    # Guardamos los resultados para que otras celdas los puedan usar.
    # .copy() crea copias independientes (evita que cambios futuros afecten lo guardado).
    ultima_imagen_rgb = img_np.copy()
    ultimo_texto      = texto_mejor
    ultimo_metodo     = metodo_ganador
    ultimos_datos     = datos_mejores.copy()

    return texto_mejor, metodo_ganador, resultados, datos_mejores


print("✅ Todas las funciones han sido definidas correctamente.")
print()
print("📌 Resumen de funciones disponibles:")
print("   preprocesar_imagen()    → convierte imagen a binario para OCR")
print("   ejecutar_ocr()          → extrae texto e información de palabras")
print("   dibujar_resultados()    → visualiza detecciones con colores de confianza")
print("   mostrar_resultados_ocr()→ imprime texto y tabla de palabras")
print("   ocr_mejor_metodo()      → compara 3 métodos y elige el mejor")
print("   pipeline_ocr_completo() → ejecuta todo el proceso de inicio a fin")

---
## 📁 CELDA 4 — Opción A: Subir una imagen desde tu computadora

### ¿Qué hace esta celda?
Crea una **interfaz gráfica interactiva** dentro del notebook con:
- Un botón para **seleccionar un archivo** de imagen desde tu disco
- Un botón para **ejecutar el OCR**
- Un área donde aparecen los **resultados**

### Formatos aceptados: `.jpg` `.jpeg` `.png` `.bmp` `.tiff`

### ¿Cómo usar?
1. Haz clic en **"Subir imagen"** y selecciona un archivo
2. Haz clic en **"Procesar OCR"**
3. Espera los resultados debajo del botón

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 4 — OCR DESDE ARCHIVO SUBIDO (ipywidgets)            ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Crear los widgets (controles de interfaz) ─────────────────
#
# widgets.FileUpload crea un botón para subir archivos.
# Parámetros:
#   accept   → extensiones de archivo permitidas (filtro del explorador de archivos)
#   multiple → False = solo un archivo a la vez

uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.bmp,.tiff',
    multiple=False,
    description='Subir imagen'
)

# widgets.Button crea un botón clickeable.
# button_style='success' le da color verde (hay: success, info, warning, danger)
boton_procesar = widgets.Button(
    description='Procesar OCR',
    button_style='success'
)

# widgets.Output crea un área donde el código puede imprimir resultados.
# Todo lo que se ejecute dentro de 'with salida_upload:' aparece en esa área.
salida_upload = widgets.Output()

# widgets.VBox organiza widgets en columna (Vertical Box).
# display() los renderiza en la pantalla del notebook.
display(widgets.VBox([uploader, boton_procesar, salida_upload]))


# ── Definir qué pasa al hacer clic en 'Procesar OCR' ──────────
#
# Los callbacks son funciones que SE EJECUTAN AUTOMÁTICAMENTE
# cuando ocurre un evento (en este caso, un clic en el botón).
# El parámetro '_' recibe información del evento, pero no la necesitamos.

def procesar_archivo_subido(_):
    global ultimo_nombre_archivo

    # 'with salida_upload:' redirige toda la salida (prints, gráficas)
    # hacia el área del widget salida_upload.
    with salida_upload:
        clear_output()   # borra el contenido anterior para no acumular resultados

        # Verificar que el usuario haya seleccionado un archivo.
        # uploader.value es vacío si no se subió nada.
        if not uploader.value:
            print("⚠️ Primero sube una imagen.")
            return   # 'return' sin valor termina la función inmediatamente

        # ── Compatibilidad entre versiones de ipywidgets ───────
        # ipywidgets puede devolver el archivo en 2 formatos distintos
        # dependiendo de la versión instalada:
        #   - versiones antiguas: tuple o list de dicts
        #   - versiones nuevas: dict con nombres como claves
        # El siguiente código maneja ambos casos.

        valor = uploader.value

        if isinstance(valor, (tuple, list)):
            # Versión antigua: valor es una lista/tupla de archivos
            archivo = valor[0]            # tomamos el primer (y único) elemento
        elif isinstance(valor, dict):
            # Versión nueva: valor es un dict {nombre_archivo: datos}
            archivo = list(valor.values())[0]   # tomamos el primer valor del dict
        else:
            print(f"⚠️ Formato inesperado en uploader.value: {type(valor)}")
            return

        # ── Extraer contenido y nombre del archivo ─────────────
        # El archivo puede ser un dict o un objeto, dependiendo de la versión.

        if isinstance(archivo, dict):
            contenido = archivo['content']            # bytes de la imagen
            nombre    = archivo.get('name', 'imagen_subida')  # nombre del archivo
            # .get() devuelve el segundo argumento si la clave no existe
        else:
            contenido = archivo.content
            nombre    = getattr(archivo, 'name', 'imagen_subida')
            # getattr(objeto, 'atributo', valor_por_defecto) es seguro:
            # si el objeto no tiene ese atributo, usa el valor por defecto

        ultimo_nombre_archivo = nombre
        print(f"📄 Archivo recibido: {nombre}")

        # ── Convertir bytes → PIL Image → NumPy array ──────────
        #
        # El archivo subido llega como 'bytes' (datos crudos de la imagen).
        # Para procesarlos necesitamos convertirlos paso a paso:
        #
        #   bytes → io.BytesIO → PIL Image → NumPy array
        #
        # io.BytesIO crea un 'archivo en memoria' a partir de los bytes,
        # que PIL puede leer igual que si fuera un archivo en disco.
        # .convert('RGB') asegura que la imagen esté en formato RGB estándar
        # (algunos PNG tienen canal alfa RGBA, que causaría errores en OpenCV).

        img_pil = Image.open(io.BytesIO(contenido)).convert('RGB')

        # np.array() convierte el objeto PIL a array NumPy.
        # Resultado: array de forma (alto, ancho, 3) con valores 0-255.
        img_np = np.array(img_pil)

        # ── Mostrar la imagen original ─────────────────────────
        plt.figure(figsize=(10, 6))
        plt.imshow(img_np)
        plt.title('Imagen original cargada')
        plt.axis('off')
        plt.show()

        # ── Ejecutar el pipeline completo de OCR ───────────────
        # Llamamos a la función principal que hace todo el trabajo.
        # Los 4 valores de retorno se guardan en variables locales.
        texto_final, metodo_ganador, resultados, datos_mejores = pipeline_ocr_completo(
            img_np,
            fuente='archivo subido',
            idioma='spa+eng',
            psm=3,
            mostrar_comparacion=True
        )

        # ── Construir tabla comparativa de los 3 métodos ───────
        # Creamos una lista de diccionarios, donde cada dict es una fila de la tabla.
        # pd.DataFrame(filas) convierte esa lista en un DataFrame.

        filas = []
        for metodo, info in resultados.items():
            filas.append({
                'Método': metodo,
                'Palabras >70% confianza': info['score'],
                'Caracteres detectados': len(info['texto'].strip()),
                'Vista previa': info['texto'][:120].replace('\n', ' ')  # primeros 120 caracteres
            })

        # display() muestra el DataFrame como tabla HTML formateada en el notebook
        display(pd.DataFrame(filas))


# ── Registrar el callback en el botón ─────────────────────────
# on_click() vincula la función al evento de clic.
# Desde ahora, cada vez que el usuario haga clic, se ejecutará procesar_archivo_subido().
boton_procesar.on_click(procesar_archivo_subido)

---
## 📷 CELDA 5 — Opción B: Captura con webcam

### ¿Qué hace esta celda?
Abre tu cámara web y muestra un **video en tiempo real** dentro del notebook.  
Al presionar **"Tomar Foto"**, captura un fotograma y le aplica OCR al instante.

### Conceptos nuevos en esta celda:
| Concepto | Explicación |
|---|---|
| **Threading** | Ejecutar dos cosas al mismo tiempo: el video Y la interfaz |
| **cv2.VideoCapture** | Objeto de OpenCV para acceder a cámaras |
| **Hilo (Thread)** | Proceso ligero que corre en paralelo al código principal |
| **Estado compartido** | Diccionario que comunica la UI con el hilo de la cámara |

> 💡 Si no encuentra tu cámara con índice `0`, cámbialo a `1` al final de la celda.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — CAPTURA CON WEBCAM LOCAL                         ║
# ╚══════════════════════════════════════════════════════════════╝

# Reimportamos para asegurar disponibilidad si se ejecuta esta celda de forma aislada.
import cv2
import pytesseract
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import threading    # Permite crear y gestionar hilos (threads)
import time         # Permite pausar la ejecución: time.sleep(segundos)
from PIL import Image
import io

# Ruta a Tesseract (misma que en la celda 2)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'


# ── Estado global de la cámara ─────────────────────────────────
#
# Usamos un diccionario como 'canal de comunicación' entre:
#   - El hilo de la cámara (que lee fotogramas continuamente)
#   - Los botones de la interfaz (que el usuario presiona)
#
# ¿Por qué un diccionario y no variables sueltas?
# Los diccionarios son MUTABLES: se pueden modificar desde cualquier hilo.
# Las variables simples en Python no siempre se comparten entre hilos correctamente.

estado = {
    'corriendo': True,   # True = el loop de cámara sigue activo
    'capturar':  False,  # True = el usuario presionó 'Tomar Foto'
    'frame':     None    # Guarda el último fotograma capturado
}


# ── Función auxiliar: OCR sobre imagen BGR ────────────────────
#
# La webcam devuelve imágenes en formato BGR (Azul, Verde, Rojo),
# el orden predeterminado de OpenCV (históricamente así era más rápido).
# Esta función hace un OCR rápido (solo con umbral Otsu, sin comparación)
# para mayor velocidad en tiempo real.

def ocr_rapido_webcam(image_bgr):
    # COLOR_BGR2GRAY convierte de BGR a escala de grises (no RGB2GRAY)
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Aplicamos umbral Otsu directamente (más rápido que comparar 3 métodos)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Ejecutamos OCR en español e inglés
    texto = pytesseract.image_to_string(thresh, lang='spa+eng')

    # .strip() elimina espacios y saltos al inicio/final
    return texto.strip() if texto.strip() else "(No se detectó texto)"


# ── Función auxiliar: convertir fotograma a bytes JPEG ────────
#
# El widget widgets.Image espera bytes en formato JPEG o PNG.
# OpenCV maneja arrays NumPy; hay que convertir para que el widget lo muestre.
#
# Proceso de conversión:
#   array BGR (OpenCV) → array RGB (PIL) → objeto PIL Image → bytes JPEG

def frame_a_bytes(frame):
    """Convierte un fotograma BGR de OpenCV a bytes JPEG para el widget."""
    # Las triples comillas al inicio son el 'docstring': descripción de la función.
    # Es buena práctica documentar siempre las funciones.

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # BGR → RGB
    pil_img = Image.fromarray(rgb)                 # array → objeto PIL
    buf = io.BytesIO()                             # buffer en memoria (como archivo temporal)
    pil_img.save(buf, format='JPEG', quality=70)   # guardar en buffer como JPEG
    #                                quality=70: compresión moderada (menor calidad, menor peso)
    return buf.getvalue()                          # retornar los bytes del JPEG


# ── Función principal de la cámara ────────────────────────────

def capturar_desde_camara(indice_camara=0):
    """
    Abre la cámara web y muestra un preview en tiempo real.
    Permite capturar fotogramas y aplicarles OCR con un clic.

    indice_camara: 0 = primera cámara del sistema, 1 = segunda cámara, etc.
    """
    global estado

    # Reiniciamos el estado para poder llamar la función varias veces
    estado = {'corriendo': True, 'capturar': False, 'frame': None}

    # ── Abrir la cámara ────────────────────────────────────────
    # cv2.VideoCapture(índice) abre la cámara especificada.
    # cv2.CAP_DSHOW es el backend DirectShow (solo Windows).
    # En Linux/macOS, puedes omitir cv2.CAP_DSHOW.

    cam = cv2.VideoCapture(indice_camara, cv2.CAP_DSHOW)

    if not cam.isOpened():   # Verificamos si la cámara se abrió correctamente
        print(f"❌ No se encontró cámara en índice {indice_camara}. Prueba con 1.")
        return

    # ── Crear widgets de interfaz ──────────────────────────────

    # widgets.Image muestra imágenes en el notebook sin recargar la celda.
    # Es más eficiente que plt.imshow() para actualizaciones frecuentes.
    img_widget = widgets.Image(
        format='jpeg',   # formato de los bytes que recibirá
        width=640,       # ancho en píxeles en la pantalla
        height=480       # alto en píxeles en la pantalla
    )

    btn_capturar  = widgets.Button(description="📷 Tomar Foto", button_style='success')
    btn_salir     = widgets.Button(description="⏹ Salir",       button_style='danger')

    # Textarea es un cuadro de texto multilínea (como un bloc de notas)
    lbl_resultado = widgets.Textarea(
        value='Aquí aparecerá el texto detectado...',
        layout=widgets.Layout(width='640px', height='120px')  # tamaño fijo del cuadro
    )

    # ── Definir callbacks de los botones ──────────────────────

    def on_capturar(b):
        # Al presionar 'Tomar Foto', activamos la bandera.
        # El hilo de la cámara revisará esta bandera y ejecutará el OCR.
        estado['capturar'] = True

    def on_salir(b):
        # Al presionar 'Salir', detenemos el loop del hilo.
        estado['corriendo'] = False

    btn_capturar.on_click(on_capturar)
    btn_salir.on_click(on_salir)

    # Mostramos todos los widgets organizados verticalmente
    display(
        widgets.VBox([
            widgets.HBox([btn_capturar, btn_salir]),  # HBox = alineación horizontal
            img_widget,
            lbl_resultado
        ])
    )

    # ── Loop de cámara en hilo separado ───────────────────────
    #
    # PROBLEMA: Si el loop de la cámara corre en el hilo principal,
    # Jupyter se bloquea y no puede procesar clics en los botones.
    #
    # SOLUCIÓN: Ejecutar el loop en un HILO SEPARADO (Thread).
    # Así el hilo principal queda libre para responder a la UI.
    #
    # Ambos hilos comparten el diccionario 'estado' para comunicarse.

    def loop_camara():
        """Función que corre en el hilo secundario: lee y muestra fotogramas."""

        while estado['corriendo']:   # continúa mientras no se presione 'Salir'

            ret, frame = cam.read()
            # cam.read() devuelve:
            #   ret   → True si el fotograma se leyó correctamente
            #   frame → el fotograma como array NumPy (BGR)

            if not ret:   # si la lectura falla (cámara desconectada, etc.)
                break

            estado['frame'] = frame   # guardamos el fotograma actual

            # Actualizamos el widget con el fotograma actual.
            # Asignar a img_widget.value actualiza la imagen mostrada.
            img_widget.value = frame_a_bytes(frame)

            # ── Verificar si se pidió captura ──────────────────
            if estado['capturar']:
                estado['capturar'] = False   # desactivamos la bandera
                lbl_resultado.value = "⏳ Procesando OCR..."

                texto = ocr_rapido_webcam(frame)   # OCR sobre el fotograma actual
                lbl_resultado.value = texto         # mostramos el texto en el Textarea

            # time.sleep() pausa el hilo N segundos.
            # 0.03 segundos ≈ 33 fps (suficiente para video fluido)
            # Sin esta pausa, el hilo usaría el 100% de la CPU.
            time.sleep(0.03)

        # Al salir del while, liberamos la cámara
        cam.release()   # cam.release() devuelve el control de la cámara al sistema

        # Notificamos al usuario
        lbl_resultado.value = lbl_resultado.value + "\n\n✅ Cámara cerrada."

    # ── Crear e iniciar el hilo ────────────────────────────────
    # threading.Thread crea un hilo que ejecutará la función 'loop_camara'
    # daemon=True: el hilo se destruye automáticamente cuando Jupyter se cierra

    hilo = threading.Thread(target=loop_camara, daemon=True)
    hilo.start()   # .start() inicia la ejecución del hilo en paralelo


# ════════════════════════════════════════════════════════════════
# EJECUTAR LA FUNCIÓN
# ════════════════════════════════════════════════════════════════
# Cambia el 0 por 1 si tu webcam principal no se detecta con índice 0.
# Esto ocurre cuando hay cámaras virtuales (OBS, VirtualCam) instaladas.

capturar_desde_camara(0)   # 0 = primera cámara detectada por el sistema

---
## 🔬 CELDA 6 — Opción C: Comparar modos PSM de Tesseract

### ¿Qué es PSM (Page Segmentation Mode)?

Tesseract necesita saber **cómo está organizado el texto** en la imagen  
para segmentarlo (separarlo en líneas, palabras, caracteres) correctamente.

| PSM | Modo | Mejor para |
|-----|------|------------|
| 3 | Automático (default) | Páginas con texto variado |
| 6 | Bloque de texto uniforme | Párrafos completos |
| 7 | Una sola línea | Títulos, subtítulos |
| 8 | Una sola palabra | Etiquetas, logos, botones |
| 11 | Texto disperso | Formularios, recibos, imágenes complejas |
| 13 | Línea sin análisis | Texto simple con formato predecible |

> ⚠️ **Requisito**: Ejecuta primero la **Opción A o B** para tener una imagen procesada.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 6 — COMPARADOR DE MODOS PSM DE TESSERACT             ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Definición de los modos PSM a comparar ────────────────────
#
# Usamos un diccionario donde:
#   clave   → número del modo PSM (entero)
#   valor   → descripción legible del modo
#
# Tesseract tiene 14 modos PSM en total (0-13).
# Aquí incluimos los 6 más relevantes para casos prácticos.

psm_configs = {
    3:  'Automático (default) — páginas normales',
    6:  'Bloque de texto uniforme — párrafos',
    7:  'Una sola línea — títulos, encabezados',
    8:  'Una sola palabra — etiquetas, logos',
    11: 'Texto disperso — imágenes complejas',
    13: 'Línea sin análisis — texto simple',
}

# ── Verificar que haya una imagen cargada previamente ─────────
#
# 'ultima_imagen_rgb is None' es True si aún no se procesó ninguna imagen.
# Esto protege el código de ejecutarse sin datos válidos.

if ultima_imagen_rgb is None:
    print('⚠️ Primero procesa una imagen con la opción A o B.')

else:
    # Preprocesamos la imagen con el mejor método encontrado anteriormente.
    # 'ultimo_metodo or adaptativo': si ultimo_metodo es cadena vacía (falsy),
    # usa 'adaptativo' como valor por defecto.
    img_preprocesada = preprocesar_imagen(ultima_imagen_rgb, metodo=ultimo_metodo or 'adaptativo')

    # ── Encabezado de la tabla de resultados ──────────────────
    print("🔬 Comparando configuraciones PSM de Tesseract:\n")

    # f-string con formato de columnas:
    # {campo:<N} → alinea a la IZQUIERDA con ancho N
    # {campo:>N} → alinea a la DERECHA con ancho N
    # Esto genera columnas alineadas como en una tabla de texto.
    print(f"{'PSM':<5} {'Palabras':<10} {'Descripción':<45} {'Texto (preview)'}")
    print('-' * 110)  # línea separadora de 110 guiones

    resultados_psm = []   # lista para guardar resultados y encontrar el ganador

    # ── Probar cada modo PSM ───────────────────────────────────
    # .items() devuelve pares (clave, valor) del diccionario.
    # 'psm' = número del modo, 'descripcion' = texto descriptivo

    for psm, descripcion in psm_configs.items():

        texto_psm, datos_psm = ejecutar_ocr(img_preprocesada, idioma='spa+eng', psm=psm)

        # [:60] toma solo los primeros 60 caracteres de la vista previa.
        # .replace('\n', ' ') reemplaza saltos de línea por espacios
        # para que el preview quepa en una sola línea de la tabla.
        preview = texto_psm.strip().replace('\n', ' ')[:60]

        num_palabras = len(datos_psm)   # total de palabras detectadas con este PSM

        # Guardamos los datos de este modo para comparar al final
        resultados_psm.append((psm, num_palabras, descripcion, preview))

        # Imprimimos la fila de la tabla con formato alineado
        print(f"{psm:<5} {num_palabras:<10} {descripcion:<45} {preview}")

    # ── Encontrar el PSM ganador ───────────────────────────────
    # max() con key=lambda busca el elemento con mayor cantidad de palabras.
    # x[1] accede al segundo elemento de cada tupla (num_palabras).
    mejor_psm = max(resultados_psm, key=lambda x: x[1])

    print(f"\n🏆 PSM con más palabras detectadas: PSM {mejor_psm[0]} → {mejor_psm[1]} palabras")

---
## 💾 CELDA 7 — Guardar el texto extraído a archivo `.txt`

### ¿Por qué exportar a .txt?
El texto extraído por OCR vive en memoria (en la variable `ultimo_texto`).  
Para que persista después de cerrar Jupyter, necesitamos guardarlo en disco.

Esta celda también incluye **metadatos** (información sobre la información):  
el nombre del archivo origen, el método ganador y las estadísticas de confianza.

> ⚠️ **Requisito**: Ejecuta primero la **Opción A o B**.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 7 — GUARDAR TEXTO EXTRAÍDO A ARCHIVO .TXT            ║
# ╚══════════════════════════════════════════════════════════════╝

# Verificamos que haya texto para guardar.
# .strip() elimina espacios; si el resultado es cadena vacía → es falsy → no guardamos.

if ultimo_texto.strip():

    nombre_archivo = 'texto_extraido_ocr.txt'
    separador      = '=' * 55   # cadena de 55 signos igual para separar secciones

    # ── Escritura del archivo ──────────────────────────────────
    #
    # open(ruta, modo, encoding) abre (o crea) un archivo.
    # Modos más comunes:
    #   'r'  → lectura (read)   → el archivo debe existir
    #   'w'  → escritura (write) → crea el archivo si no existe; borra si existe
    #   'a'  → agregar (append) → agrega al final sin borrar el contenido previo
    #
    # encoding='utf-8' es crucial para que caracteres especiales (tildes,
    # eñes, emojis) se guarden correctamente en el archivo.
    #
    # 'with open(...) as f:' es el patrón recomendado para manejo de archivos.
    # Garantiza que el archivo se CIERRA automáticamente al terminar el bloque,
    # incluso si ocurre un error. Sin 'with', podrías dejar archivos abiertos.

    with open(nombre_archivo, 'w', encoding='utf-8') as f:

        # print(..., file=f) redirige la salida al archivo en lugar de la pantalla.
        # Es equivalente a f.write(...) pero más legible.
        print(separador, file=f)
        print('  TEXTO EXTRAÍDO POR OCR', file=f)
        print(separador, file=f)

        # Solo imprimimos la línea de origen si tenemos el nombre del archivo.
        # Las cadenas no vacías son 'truthy' en Python.
        if ultimo_nombre_archivo:
            print(f'Archivo origen : {ultimo_nombre_archivo}', file=f)

        print(f'Método ganador : {ultimo_metodo}', file=f)

        # El texto OCR puede tener múltiples líneas; lo escribimos tal cual.
        print(ultimo_texto.strip(), file=f)
        print(separador, file=f)

        # Estadísticas solo si hay datos disponibles
        if not ultimos_datos.empty:
            print(f'Palabras detectadas : {len(ultimos_datos)}', file=f)

            # .mean() calcula el promedio de la columna 'conf'
            # :.1f formatea el resultado con 1 decimal
            print(f'Confianza promedio  : {ultimos_datos["conf"].mean():.1f}%', file=f)

    # Confirmamos al usuario dónde se guardó el archivo.
    print(f"✅ Archivo guardado correctamente: {nombre_archivo}")
    print('📂 Se guardó en la misma carpeta de trabajo del notebook.')

else:
    # Si no hay texto, orientamos al usuario sobre qué paso falta.
    print('⚠️ Todavía no hay texto para guardar.')
    print('   Ejecuta primero el OCR sobre una imagen (Opción A o B).')

---
## 📋 CELDA 8 — Resumen pedagógico y ejercicios propuestos

### 🗺️ ¿Qué aprendiste en este notebook?

| Tema | Concepto | Dónde aparece |
|---|---|---|
| Python básico | Importaciones, funciones, variables globales | Celdas 1 y 2 |
| NumPy | Arrays como representación de imágenes | Toda la sección 3 |
| OpenCV | cvtColor, threshold, resize, GaussianBlur | Función `preprocesar_imagen` |
| Tesseract | PSM, idioma, image_to_string, image_to_data | Función `ejecutar_ocr` |
| Matplotlib | subplots, imshow, legend, patches | Función `dibujar_resultados` |
| pandas | DataFrame, filtrado, sort_values, mean | Toda la sección de resultados |
| ipywidgets | FileUpload, Button, Output, VBox, HBox | Celdas 4 y 5 |
| Threading | Thread, daemon, hilo paralelo | Celda 5 |
| Archivos | open, with, encoding, write | Celda 7 |

---

### 🚀 Ejercicios propuestos para practicar

**Nivel básico:**
1. Cambia el umbral de confianza de `70%` a `60%` en `ocr_mejor_metodo` y observa si cambia el método ganador.
2. Agrega el modo `PSM 4` (columna de texto) al comparador de la Celda 6.
3. Modifica el color de los rectángulos de **confianza baja** a azul en lugar de rojo.

**Nivel intermedio:**
4. Agrega un widget `widgets.IntSlider` para que el usuario ajuste el umbral de confianza sin modificar el código.
5. Agrega una función que cuente cuántas veces aparece cada palabra en el texto extraído (frecuencia de palabras).
6. Modifica la exportación `.txt` para que también guarde la tabla de confianza por palabra.

**Nivel avanzado:**
7. Implementa un preprocesamiento adicional: **corrección de rotación** con `cv2.minAreaRect` antes del OCR.
8. Agrega soporte para procesar un **PDF** completo página por página usando `pdf2image`.
9. Crea una función que detecte si el texto tiene **orientación vertical** y lo rote automáticamente.

---

### 💡 Consejos para mejores resultados en OCR

```
✅ Iluminación uniforme sin sombras
✅ Texto en foco (no borroso)
✅ Contraste alto entre texto y fondo
✅ Texto horizontal (no rotado)
✅ Resolución mínima recomendada: 300 DPI
❌ Evitar fondos con patrones o texturas
❌ Evitar reflejos sobre el texto
❌ Evitar texto muy pequeño (menor a 12pt)
```

---

### 🔄 Pipeline completo resumido

```
IMAGEN DE ENTRADA (JPG / PNG / webcam)
           ↓
    CONVERSIÓN A GRISES
    (cv2.COLOR_RGB2GRAY)
           ↓
    ESCALADO si < 600px
    (cv2.resize + INTER_CUBIC)
           ↓
    SUAVIZADO (GaussianBlur 3×3)
           ↓
    ┌───── UMBRALIZACIÓN ──────┐
    │  Ninguno │ Otsu │ Adapt. │
    └──────────────────────────┘
           ↓
    OCR CON TESSERACT
    (image_to_string + image_to_data)
           ↓
    SELECCIÓN AUTOMÁTICA
    (max score con conf > 70%)
           ↓
    VISUALIZACIÓN + EXPORTACIÓN
```